In [ ]:
!git pull
from openai import OpenAI
import time

from api import PROJECT_KEY

client = OpenAI(
    api_key=PROJECT_KEY,
    base_url="https://openrouter.ai/api/v1",
)

def chat(model, messages, temperature=0):
    return client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    ).choices[0].message.content
client = OpenAI(
    api_key=PROJECT_KEY,
    base_url="https://openrouter.ai/api/v1",
)


# -------------------------ask in german / french ->>english translation!

with open("Na'vi Files/training.txt", "r") as f:
    training = f.read()

with open("test3.txt", "r") as f:
    test = f.read()

prompt = f"""

Du lernst eine konstruierte Sprache.
Lerne von diesen Beispielen mit der Formattierung
Na'vi Satz | Englische Übersetzung :

{training}

Jetzt übersetze die folgenden Sätze akkurat.
Gib deine Antwort im gleichen Format wie die Training-Sätze, Na'vi Satz | Deine Englische Übersetzung.
Gib keine Erklärung, sondern nur die Übersetzung.

Alaksi srak, ma frapo?
Am’aluke snayaytx Sawtute, yayora’ Na’vi.
Awnga kelku si nuä ayram alusìng.
Ayfo solop ìlä hilvan fa uran.
Aylì’fya yawne leru oer takrra ’eveng lamu.




"""

# QWEN--------------------------------------------------------------------

def query_qwen(prompt):
    return chat(
        model="qwen/qwen-2.5-72b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )

# LLAMA --------------------------------------------------------

def query_llama(prompt):
    return chat(
        model="meta-llama/llama-3.1-8b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )

models = {
    #"gpt4o": query_gpt,
    "qwen": query_qwen,
    "llama": query_llama
}

for name, function in models.items():

    print(f"Running {name}...")

    output = function(prompt)

    with open(f"results/{name}colab.txt", "w") as f:
        f.write(output)
        print(f"Saved {name}colab.txt")

Already up to date.
Running qwen...
Saved qwencolab.txt
Running llama...
Saved llamacolab.txt


In [ ]:
%pip install -q openai sacrebleu

from openai import OpenAI
from sacrebleu.metrics import CHRF
import os
from api import PROJECT_KEY
import time


MODELS = {
    #"gpt4o": "openai/gpt-4o",
    "gemini": "google/gemini-2.5-pro",
    "qwen": "qwen/qwen3-30b-a3b",
    "llama": "meta-llama/llama-3.1-8b-instruct"
}

TRAINING_FILE = "Na'vi Files/training.txt"
TEST_FILE = "test4.txt"
REFERENCE_FILE = "test3SOLUTIONS.txt"
OUTPUT_DIR = "results"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "BatchTranslations.txt")



client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=PROJECT_KEY,
)


with open(TRAINING_FILE, "r", encoding="utf-8") as f:
    training = f.read()

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_sentences = [
        line.strip()
        for line in f
        if line.strip()
    ]

# -------------------------BATCHES OF 5

def chunk(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

#------------------------------------------PROMPT

def build_prompt(training, batch):

    sentences = "\n".join(batch)

    return f"""

You are learning a constructed language.
Study these examples which are formatted like 
Na'vi sentence | English translation :

{training}

Now accurately translate the following sentences.
Give your response in the same format as the training sentences, Na'vi Sentence | Your english translation.
Do not explain, only give the translations.

Sentences:

{sentences}
"""

# -----------------------Query model

import time

def translate_batch(batch, model):

    prompt = build_prompt(training, batch)

    for attempt in range(3):

        try:
            response = client.chat.completions.create(
                model=model,
                temperature=0,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            content = response.choices[0].message.content

            if content is None:
                raise RuntimeError("Model returned no content.")

            return content.strip()

        except Exception as e:
            print(f"Attempt {attempt+1}/3 failed: {e}")

            if attempt == 2:
                raise

            time.sleep(5)

# -------------------------------------------------- Translaters


os.makedirs(OUTPUT_DIR, exist_ok=True)

for model_name, model_id in MODELS.items():

    print(f"\n{model_name} :")

    all_translations = []

    for batch_number, batch in enumerate(chunk(test_sentences, 5), start=1):

        print(f"Batch {batch_number}")

        output = translate_batch(batch, model_id)

        lines = [
            line.strip()
            for line in output.splitlines()
            if line.strip()
        ]

        all_translations.extend(lines)

    output_file = os.path.join(
        OUTPUT_DIR,
        f"{model_name}_translations.txt"
    )

    with open(output_file, "w", encoding="utf-8") as f:
        for line in all_translations:
            f.write(line + "\n")

    print(f"Saved {output_file}")

    # ---------- chrF evaluation ----------
    if REFERENCE_FILE is not None:

        references = []

        with open(REFERENCE_FILE, "r", encoding="utf-8") as f:
            for line in f:

                line = line.strip()

                if not line:
                    continue

                if "|" in line:
                    references.append(line.split("|", 1)[1].strip())
                else:
                    references.append(line)

        if len(references) == len(all_translations):

            metric = CHRF()
            score = metric.corpus_score(
                all_translations,
                [references]
            )

            print(f"{model_name} chrF: {score.score:.2f}")

        else:
            print("Reference and prediction counts differ.")

/bin/bash: line 1: pull: command not found


ModuleNotFoundError: No module named 'sacrebleu'

In [3]:
%cd BachPrakt/




/content/BachPrakt


In [16]:
with open("results/llamacolab.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text)

Here are the translations:

Alaksi srak, ma frapo? | Have you ever been to the forest, but not seen the forest?
Am’aluke snayaytx Sawtute, yayora’ Na’vi. | I'm glad to see Sawtute, the Na'vi language.
Awnga kelku si nuä ayram alusìng. | We live on the other side of the river.
Ayfo solop ìlä hilvan fa uran. | I will only talk to you in the language of the forest.
Aylì’fya yawne leru oer takrra ’eveng lamu. | I'm very happy that you're learning the language of the forest.
